# Assignment 2: Exercise 1

To improve the learning stability, introduce (mini) batched learning.
(See Unit 3 Page 95 of the lecture for guidance). Compare the convergence behavior of batched
learning to online learning, dig deeper than just reporting the slowdown in convergence due to
fewer update steps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Optional
from pyMLaux import plot_2d_prediction

## 1. Setup
First I will reuse the methods from the exercise, that are necessary.

### 1.1 Utilities

In [ ]:
def one_hot(y: np.ndarray, num_classes: int) -> np.ndarray:
    oh = np.zeros((y.size, num_classes))
    oh[np.arange(y.size), y] = 1.0
    return oh

# Initialization helpers

# Activations and their derivatives

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def dsigmoid(z):
    s = sigmoid(z)
    return s * (1 - s)

# Stable softmax + cross-entropy

def softmax(logits: np.ndarray) -> np.ndarray:
    z = logits - logits.max(axis=1, keepdims=True)
    expz = np.exp(z)
    return expz / expz.sum(axis=1, keepdims=True)

def binary_cross_entropy(yhat: float, y_true: float, eps: float = 1e-12) -> float:
    yhat_clipped = np.clip(yhat, eps, 1.0)
    return -(y_true * np.log(yhat_clipped) + (1 - y_true) * np.log(1 - yhat_clipped))

def cross_entropy(probs: np.ndarray, y_true_oh: np.ndarray, eps: float = 1e-12) -> float:
    # Avoid log(0)
    p = np.clip(probs, eps, 1.0)
    return -np.mean(np.sum(y_true_oh * np.log(p), axis=1))


### 1.2. MLP Class

In [ ]:
class MLP:
    def __init__(
        self,
        input_dim: int,
        layer_sizes: List[int],
        num_classes: int,
        seed: Optional[int] = None,
    ):
        self.rng = np.random.default_rng(seed)
        self.shapes = []

        self.act = sigmoid
        self.dact = dsigmoid

        # layer dimensions
        dims = [input_dim] + layer_sizes + [num_classes]
        self.L = len(dims) - 1

        # params init
        self.W = {}
        self.b = {}
        for k in range(1, self.L + 1):
            fan_in, fan_out = dims[k-1], dims[k]
            Wk = np.random.randn(fan_in, fan_out)
            bk = np.zeros((1, fan_out))
            self.W[k] = Wk
            self.b[k] = bk
            self.shapes.append((fan_in, fan_out))

    def forward(self, X: np.ndarray, train: bool = True):
        caches = []
        A = X
        for k in range(1, self.L + 1):
            Z = A @ self.W[k] + self.b[k]
            if k == self.L:
                A_next = softmax(Z)
            else:
                A_next = self.act(Z)
            caches.append((Z, A, self.W[k], self.b[k]))
            A = A_next
        return A, caches

    def backward(self, probs: np.ndarray, y_onehot: np.ndarray, caches: List[tuple]):
        grads_W = {}
        grads_b = {}
        m = y_onehot.shape[0]

        # Initial gradient at output: softmax + cross-entropy
        dA = (probs - y_onehot) / m  # shape (m, C)

        for k in reversed(range(1, self.L + 1)):
            Z, A_prev, Wk, bk = caches[k-1]
            dZ = dA  # start with upstream gradient
            if k != self.L:
                # back through activation
                dZ = dZ * self.dact(Z)

            # linear layer grads
            dW = A_prev.T @ dZ
            db = np.sum(dZ, axis=0, keepdims=True)
            dA = dZ @ Wk.T

            grads_W[k] = dW
            grads_b[k] = db
        return grads_W, grads_b

    def step(self, grads_W: Dict[int, np.ndarray], grads_b: Dict[int, np.ndarray], lr: float):
        for k in range(1, self.L + 1):
            self.W[k] -= lr * grads_W[k]
            self.b[k] -= lr * grads_b[k]

    def compute_loss(self, probs: np.ndarray, y_onehot: np.ndarray):
        loss = cross_entropy(probs, y_onehot)
        return loss

    def predict(self, X: np.ndarray):
        probs, _ = self.forward(X, train=False)
        return np.argmax(probs, axis=1), probs


### 1.3 Training function

In [ ]:
def accuracy(y_true, y_pred):
    return (y_true == y_pred).mean()


def train_online_learning(
    model: MLP,
    X: np.ndarray,
    y: np.ndarray,
    epochs: int = 200,
    lr: float = 1e-0,
    seed: Optional[int] = 123,
    verbose: bool = True
):
    np.random.default_rng(seed)
    num_classes = int(y.max()) + 1
    y_oh = one_hot(y, num_classes)
    history = {"loss": [], "acc": []}

    for epoch in range(1, epochs + 1):
        # Training epoch
        r = np.arange(X.shape[0])
        np.random.shuffle(r)
        for i in r:
            yb_oh = one_hot(y[i].reshape(-1,1), num_classes)
            probs, caches = model.forward(X[i, :].reshape(-1,X.shape[1]), train=True)
            grads_W, grads_b = model.backward(probs, yb_oh, caches)
            model.step(grads_W, grads_b, lr=lr)

        # Evaluate on full training set (for simplicity)
        probs_full, _ = model.forward(X, train=False)
        loss = model.compute_loss(probs_full, y_oh)
        y_pred = np.argmax(probs_full, axis=1)
        acc = accuracy(y, y_pred)
        history["loss"].append(loss)
        history["acc"].append(acc)

        if verbose and (epoch % max(1, epochs // 10) == 0 or epoch <= 5):
            print(f"Epoch {epoch:4d} | loss={loss:.4f} | acc={acc*100:.2f}%")
    return history


## 2. Introducing batched learning

- The core difference between these two learning methods is, that online learning updates w for each sample individually. Batch learning sums up the local updates for all samples, then updates w. This tries to reduce the error for the entire training set, not just for the specific sample
- I first locate the code in the lesson, where the online learning is used. Then my goal is to replace that chunk with the batch learning method
    - The `for i in r` loop (and the shuffling needed for this method) does the online learning. In the following code, i replace that loop to do batch learning
    - The new method needs a new hyperparameter `batch_size`



In [ ]:
def train_batch_learning(
    model: MLP,
    X: np.ndarray,
    y: np.ndarray,
    epochs: int = 200,
    batch_size: int = 32, #new hyperparameter
    lr: float = 1e-0,
    seed: Optional[int] = 123,
    verbose: bool = True
):
    np.random.default_rng(seed)
    num_classes = int(y.max()) + 1
    y_oh = one_hot(y, num_classes)
    history = {"loss": [], "acc": []}

    for epoch in range(1, epochs + 1):

        # Training epoch
        r = np.arange(X.shape[0])
        np.random.shuffle(r)

        # Here starts the difference to online-learning
        # Loops through the datasets in steps of batch_size
        for start_idx in range(0, X.shape[0], batch_size):
            # Slices the vector into the batch and selects the targets
            batch_idx = r[start_idx : start_idx + batch_size]
            Xb = X[batch_idx]
            yb_oh = y_oh[batch_idx]

            # runs the forward for the entire batch
            probs, caches = model.forward(Xb, train=True)

            # computes the gradients from the whole batch and updates the weights and biases per mini batch
            grads_W, grads_b = model.backward(probs, yb_oh, caches)
            model.step(grads_W, grads_b, lr=lr)

        # Evaluate on full training set (for simplicity)
        probs_full, _ = model.forward(X, train=False)
        loss = model.compute_loss(probs_full, y_oh)
        y_pred = np.argmax(probs_full, axis=1)
        acc = accuracy(y, y_pred)
        history["loss"].append(loss)
        history["acc"].append(acc)

        if verbose and (epoch % max(1, epochs // 10) == 0 or epoch <= 5):
            print(f"Epoch {epoch:4d} | loss={loss:.4f} | acc={acc*100:.2f}%")
    return history


## 3. Compare the convergence behavior of batched learning to online learning
- dig deeper than just reporting the slowdown in convergence due to fewer update steps.